# 13 — Direction 1, step D: speed of the differentiable replica on a GPU

`hvac_controller/d1/bench.py`, one process per scenario, run one after the other so each gets the whole GPU. For each
scenario it builds the replica once (spec, ROM, exogenous series; CPU), then:

| part | what | gate (D1 doc §5, step 4) |
|---|---|---|
| accuracy | 30 days of structured random actions (8 seeds): torch float64 / float32 on the GPU (Jacobi air solve) vs the NumPy float64 replica with the exact LU solve (step C's configuration); a batch row vs a single run | float32 zone max ≤ 0.01 K, reward sum ≤ 0.1 %; batch independence ≤ 1e-4 K |
| forward | no-grad env-steps/s at B = 1 … 4096, eager and `torch.compile`d | reported |
| forward + backward | windows h = 32, 128 on the relaxed plant, B = 64 … 4096, eager; h = 32 compiled with two Inductor settings (the default failed in Triton on the first run): env-steps/s, peak memory, finite gradients | **≥ 1e5 env-steps/s at h = 32 on the largest building (Z = 85), peak < 12 GB** |
| breakdown | ms per call of decode, plant.pre, ROM step (its air solve: Jacobi vs LU), plant.post at B = 1024; Jacobi spectral radius met vs the bound q the sweep count is set from | reported |
| compile diagnostics | one gradient window, eager vs `aot_eager` vs Inductor: reward gap, which gradients are non-finite at which step | reported |

**Setup:** accelerator **GPU T4** (Triton needs compute capability ≥ 7.0, so not P100); Internet on (clones the repo);
attach the competition data. Expected ≈ 60–90 min (ROM of the 85-zone building ≈ 2–4 min; one compile per batch size). Output:
`/kaggle/working/d1_speed.zip`.

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "direction1"
SCENARIOS = ["building_train_1/atlanta_2", "building_train_2/phoenix_6", "building_train_8/fairbanks_8"]
STEPS = 8640       # accuracy check: one full episode
COMPILE = "aot_eager,inductor,inductor_no_persistent"  # compile variants (bench.COMPILE_VARIANTS); "" for none

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

import torch
assert torch.cuda.is_available(), "no GPU: set the accelerator to GPU T4"
print(torch.__version__, torch.cuda.get_device_name(), torch.cuda.get_device_capability())

REPO = Path("/tmp/hvac-controller")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(COMMIT)

SHIPPED = next(p for p in sorted(Path("/kaggle/input").rglob("training_scenarios"))
               if (p / "building_train_1" / "atlanta_2" / "config.gin").exists())
shutil.copytree(SHIPPED, REPO / "smart_buildings" / "training_scenarios", dirs_exist_ok=True)

In [ ]:
OUT = Path("/kaggle/working/d1_speed")
OUT.mkdir(parents=True, exist_ok=True)
env = dict(os.environ, PYTHONPATH=str(REPO))
tags = [s.replace("/", "__") for s in SCENARIOS]
t0 = time.time()
for scenario, tag in zip(SCENARIOS, tags):
    (OUT / tag).mkdir(exist_ok=True)
    cfg = REPO / "smart_buildings" / "training_scenarios" / scenario / "config.gin"
    cmd = [sys.executable, "-m", "hvac_controller.d1.bench", str(OUT / tag), "--config", str(cfg),
           "--steps", str(STEPS), "--compile", COMPILE]
    with open(OUT / tag / "log.txt", "w") as log:
        rc = subprocess.run(cmd, cwd=REPO, env=env, stdout=log, stderr=subprocess.STDOUT).returncode
    text = (OUT / tag / "log.txt").read_text()
    print(f"[{time.time() - t0:5.0f}s] {tag} exit {rc}")
    print(text[-3000:] if "Traceback" not in text else text[-8000:])

In [ ]:
reports = {}
for tag in tags:
    f = OUT / tag / "report.json"
    reports[tag] = json.loads(f.read_text()) if f.exists() else {"error": "no report (see log above)"}
(OUT / "reports.json").write_text(json.dumps({"commit": COMMIT, "reports": reports}, indent=1))
with zipfile.ZipFile("/kaggle/working/d1_speed.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(OUT.parent))
print("wrote /kaggle/working/d1_speed.zip")


def rate(d):
    return "error" if "error" in d else f"{d['env_steps_per_s']:,.0f}"


acc_ok, best = [], {}
for tag, r in reports.items():
    if "setup" not in r:
        print(tag, "ERROR", r.get("error"))
        continue
    s = r["setup"]
    print(f"\n{tag}: Z={r['n_zones']} V={r['n_vavs']} r={r['rom_order']} on {r['device_name']} (torch {r['torch']}) | setup "
          f"load {s['load_env_s']:.0f}s spec {s['spec_s']:.0f}s ROM {s['rom_s']:.0f}s exogenous {s['exo_s']:.0f}s")
    if "accuracy" in r:
        a = r["accuracy"]
        f64, f32, bi = a["float64"], a["float32"], a["batch_independence_float32"]
        print(f"  accuracy ({a['steps']} steps, {a['rows']} rows; NumPy LU {a['numpy_s']:.0f}s; Jacobi sweeps f64 {f64.get('sweeps')}"
              f" f32 {f32.get('sweeps')}): float64 zone {f64['zone_max_k']:.1e} K"
              f" reward sum {f64['reward_sum_rel_max']:.1e} | float32 zone {f32['zone_max_k']:.1e} K reward sum "
              f"{f32['reward_sum_rel_max']:.1e} step {f32['reward_step_max_abs']:.1e} | batch independence {bi['zone_max_k']:.1e} K")
        acc_ok.append((tag, f32["zone_max_k"] <= 0.01 and f32["reward_sum_rel_max"] <= 1e-3 and bi["zone_max_k"] <= 1e-4))
    for key in [k for k in r if k.startswith("forward")]:
        print(f"  {key:>22} env-steps/s: " + "  ".join(f"B{b} {rate(v)}" for b, v in r[key].items()))
    detail, detail_compiled = None, False  # show a compile failure's traceback before any other
    for key in [k for k in r if k.startswith("backward")]:
        for k, v in r[key].items():
            if "error" in v:
                print(f"  {key:>22} {k:>10}: {v['error'][:120]}")
                if detail is None or ("compiled" in key and not detail_compiled):
                    detail, detail_compiled = v.get("detail"), "compiled" in key
                continue
            print(f"  {key:>22} {k:>10}: {v['env_steps_per_s']:>10,.0f} env-steps/s  {v['s_per_window']*1e3:7.1f} ms/window"
                  f"  peak {v['peak_mem_gb']:.2f} GB  finite {v['grad_finite']}  (first {v['first_window_s']:.1f}s)")
            if v["h"] == 32 and v["grad_finite"] and v["peak_mem_gb"] < 12.0:
                best[tag] = max(best.get(tag, (0, ""))[0], v["env_steps_per_s"]), r["n_vavs"]
    if "breakdown" in r:
        b = r["breakdown"]
        print(f"  breakdown B{b['B']} (ms): step {b['step_ms']:.2f} = decode {b['decode_ms']:.2f} + pre {b['pre_ms']:.2f}"
              f" + ROM {b['rom_ms']:.2f} (air solve {b['air_ms']:.3f} with {b['sweeps']} sweeps, LU {b['lu_ms']:.2f};"
              f" gap {b['air_vs_lu_rel']:.1e}) + post {b['post_ms']:.2f}")
        sd = b["solve_diag"]
        print(f"  air-side system: Jacobi rho max {sd['rho_max']:.3g} (running median {sd['rho_running_median']:.3g},"
              f" full air {sd['rho_full_air']:.3g}), inf-norm {sd['inf_norm_max']:.3g}; bound q at g_max {b['q_bound']:.3g}")
    if "compile_diagnostics" in r:
        cd = r["compile_diagnostics"]
        print(f"  compile diagnostics (B{cd['B']}, h{cd['h']}; eager gradients finite {cd['eager_grad_finite']}):")
        for name, v in cd.items():
            if not isinstance(v, dict):
                continue
            if "error" in v:
                print(f"    {name:>22}: {v['error'][:150]}")
            else:
                print(f"    {name:>22}: reward gap {v['reward_max_abs_diff']:.1e} (finite {v['reward_finite']}) | non-finite grads"
                      f" cont {v['grad_cont_nonfinite']} disc {v['grad_disc_nonfinite']} at steps {v['steps_with_nonfinite']}"
                      f" in {v['actions_with_nonfinite'][:8]} | finite grad gap {v['grad_cont_max_abs_diff_finite']}"
                      f" (scale {v['grad_cont_scale']:.1e})")
    if detail:
        print("  first failure, tail of its traceback:\n" + detail[-2500:])
print(f"\nGATE accuracy (float32 zone <= 0.01 K, reward sum <= 0.1 %, batch <= 1e-4 K): "
      f"{sum(ok for _, ok in acc_ok)}/{len(acc_ok)} pass")
if best:
    tag = max(best, key=lambda t: best[t][1])
    print(f"GATE speed (largest building {tag}, h=32, peak < 12 GB): {best[tag][0]:,.0f} env-steps/s ->",
          "PASS" if best[tag][0] >= 1e5 else "FAIL", "(target 1e5)")